# Prophage Deep-Learning Validation & Functional Profiling
### geNomad + CheckV + ESM2 — Free Google Colab (T4) pipeline

**Purpose.** This notebook adds an orthogonal, deep-learning-based validation layer on top of your existing PhiSpy prophage calls, and a protein-language-model functional layer for the prophage cargo genes. It is designed specifically to:

1. Resolve the manuscript's own stated limitation — *"PhiSpy-predicted regions should be interpreted as putative prophages until their structural integrity ... are experimentally established"* — using **CheckV** (completeness/contamination scoring) and **geNomad** (independent deep-learning viral classifier).
2. Add a genuinely modern GPU layer: **ESM2 protein language model embeddings** of prophage-encoded proteins, used to provide candidate functional annotations for prophage-associated proteins and compare relatedness among validated prophage regions.

**Fits the free Colab tier.** ~65 prophage regions across 20 genomes, a few thousand encoded proteins total. On a free T4 (16 GB VRAM, ~12 GB RAM, ~12 h session cap, ~90 min idle timeout) the full run (DB downloads + geNomad + CheckV + ESM2 on all regions) is expected to take **roughly 1.5–3 hours** of active compute, well inside one session.

**What you need to upload:** just the two zip files you already have —
`Final_Genomes.zip` and `PhiSpy_Results.zip`. Cell 2 gives you a direct upload widget (no Google Drive needed) and handles the nested-folder structure they come in.

*(Verified against your actual uploaded files: 20/20 genomes have `genomic.fna` + `genomic.gbff`; 20/20 have PhiSpy prophage coordinate files; row counts per genome reproduce Figure 1 exactly, 65/65 regions total. Two harmless extras are auto-ignored: an empty `WCFS1` folder in `Final_Genomes` — GCF_000203855.3 already covers that strain under its accession name — and a stray, empty `GCF_001704315.1` folder in `PhiSpy_Results` left over from an aborted run of DF's real accession, GCF_001704335.1.)*

**Storage note.** Everything runs on the Colab machine's local disk (`/content/...`), not Drive — nothing here reads from or writes to your Drive during the run. The very last cell zips up all results and gives you a **download button**; from there it's your call whether to drop that zip into Drive yourself.

**Because there's no Drive checkpointing:** if your runtime disconnects mid-run (free-tier sessions can be interrupted), you'll need to re-run from Cell 2. The DB downloads (Cell 4) and ESM2 embeddings (Cell 9) still checkpoint to local disk *within* a single session, so a re-run of the same cell won't redo finished work — but a fresh runtime starts from zero. For ~65 regions this is a 1.5–3 hour job, so that's a minor risk, not a major one.


## 1. Runtime check
Confirm you're on a GPU runtime: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")


## 2. Upload your zip files and unzip (local disk, no Drive)

Run this cell, then use the **file picker that appears** to select `Final_Genomes.zip` and `PhiSpy_Results.zip` from your computer (upload both together or one at a time — the cell handles either). Files are saved to the Colab machine's local disk.


In [ ]:
from google.colab import files
import os

UPLOAD_DIR = "/content/uploaded_zips"
os.makedirs(UPLOAD_DIR, exist_ok=True)

print("Select Final_Genomes.zip and PhiSpy_Results.zip (you can select both at once):")
uploaded = files.upload()

for fname in uploaded:
    dest = f"{UPLOAD_DIR}/{fname}"
    os.rename(fname, dest) if os.path.exists(fname) else None
    print(f"Saved: {dest}")


In [ ]:
import zipfile

# Working dirs -- everything local to this Colab session
WORK = "/content/work"
OUT  = "/content/outputs"          # all final results land here
DB   = "/content/databases"        # geNomad/CheckV DBs (re-downloaded each fresh session)
UNZIP_DIR = "/content/inputs"

for d in [WORK, OUT, DB, UNZIP_DIR]:
    os.makedirs(d, exist_ok=True)

def find_or_unzip(zip_name, extract_subdir):
    # Unzip {zip_name} into UNZIP_DIR/{extract_subdir} if not already done,
    # then return the actual directory that contains the per-genome folders
    # (your zips have an extra nested folder level, e.g. Final_Genomes/Final_Genomes/<acc>/...).
    target = f"{UNZIP_DIR}/{extract_subdir}"
    if not os.path.isdir(target) or not os.listdir(target):
        zpath = f"{UPLOAD_DIR}/{zip_name}"
        assert os.path.exists(zpath), f"Missing {zpath} -- re-run Cell 2 and upload {zip_name}"
        with zipfile.ZipFile(zpath) as zf:
            zf.extractall(target)
    cur = target
    while True:
        entries = [e for e in os.listdir(cur) if not e.startswith("__MACOSX")]
        subdirs = [e for e in entries if os.path.isdir(os.path.join(cur, e))]
        if len(subdirs) == 1 and not any(e.startswith(("GCA_", "GCF_")) for e in entries):
            cur = os.path.join(cur, subdirs[0])
        else:
            break
    return cur

GENOMES_DIR = find_or_unzip("Final_Genomes.zip", "Final_Genomes")
PHISPY_DIR  = find_or_unzip("PhiSpy_Results.zip", "PhiSpy_Results")

# Only real accession folders -- ignores stray/empty folders (e.g. WCFS1) and loose script files
accessions = sorted(d for d in os.listdir(GENOMES_DIR)
                     if os.path.isdir(os.path.join(GENOMES_DIR, d)) and d.startswith(("GCA_", "GCF_"))
                     and os.path.exists(os.path.join(GENOMES_DIR, d, "genomic.fna")))

print(f"Resolved GENOMES_DIR -> {GENOMES_DIR}")
print(f"Resolved PHISPY_DIR  -> {PHISPY_DIR}")
print(f"Found {len(accessions)} usable genome accessions")
print(accessions)


## 3. Install geNomad, CheckV, and ESM2

- `genomad` and `checkv` are pip-installable (no conda needed on Colab).
- ESM2 comes via `fair-esm` (Meta's official package) — smaller/faster than pulling the full `transformers` stack.
- Model choice: **esm2_t12_35M_UR50D** by default (fast, fits easily on a T4, plenty of resolution for clustering cargo proteins at this scale). Swap to `esm2_t30_150M_UR50D` in Cell 8 if you want higher-resolution embeddings and have time budget — still comfortably free-tier.


In [ ]:
!pip install -q genomad checkv fair-esm biopython


## 4. Download reference databases

geNomad's DB is ~2.5 GB, CheckV's is ~750 MB. Both download to local disk. Since there's no Drive caching, **this happens fresh each session** (~5–10 min on Colab's network) — the cell still checks for an existing local copy first, so re-running it after a disconnect-and-reconnect within the same runtime won't re-download.


In [ ]:
import os

genomad_db = f"{DB}/genomad_db"
checkv_db  = f"{DB}/checkv-db"

if not os.path.isdir(genomad_db):
    !genomad download-database {DB}
else:
    print("geNomad DB already present, skipping download")

if not os.path.isdir(checkv_db):
    !checkv download_database {DB}
    # checkv creates a versioned folder like checkv-db-v1.5 -- normalize the name
    import glob, shutil
    hits = glob.glob(f"{DB}/checkv-db-v*")
    if hits and not os.path.isdir(checkv_db):
        shutil.move(hits[0], checkv_db)
else:
    print("CheckV DB already present, skipping download")


## 5. Extract prophage nucleotide FASTA regions

Reads each genome's PhiSpy coordinates file and slices the corresponding region out of `genomic.fna`, writing one combined multi-FASTA. Header format: `>{accession}__pp{n}__{start}-{end}` — this ID scheme is reused through every downstream step so results always map back to a specific genome + prophage call.

**Format notes (confirmed against your actual PhiSpy output):**
- The coordinate files are PhiSpy's raw `prophage_coordinates.tsv` — **11 columns, no header row**: `prophage_id, contig, start, stop, attL_start, attL_stop, attR_start, attR_stop, attL_seq, attR_seq, note`. We only need columns 2–4 (contig, start, stop).
- Filenames are usually `{accession}_prophage_coordinates.tsv`, but one genome in your set (GCF_000203855.3 / WCFS1) was run with the plain name `prophage_coordinates.tsv` — the cell below checks both patterns.
- Contig IDs in the coordinates file (e.g. `CP005942.2`) match the FASTA header's first token exactly, so no fuzzy matching is needed — but a fallback is kept in case a future genome doesn't match cleanly.


In [ ]:
from Bio import SeqIO
import pandas as pd
import os

COORD_COLS = ["prophage_id", "contig", "start", "stop",
              "attL_start", "attL_stop", "attR_start", "attR_stop",
              "attL_seq", "attR_seq", "note"]

def resolve_coord_file(acc):
    candidates = [
        f"{PHISPY_DIR}/{acc}/{acc}_prophage_coordinates.tsv",
        f"{PHISPY_DIR}/{acc}/prophage_coordinates.tsv",
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    return None

prophage_records = []
skipped = []

combined_fasta_path = f"{WORK}/prophage_regions.fna"
with open(combined_fasta_path, "w") as out_fh:
    for acc in accessions:
        coord_file = resolve_coord_file(acc)
        fna_file   = f"{GENOMES_DIR}/{acc}/genomic.fna"
        if coord_file is None or not os.path.exists(fna_file):
            skipped.append((acc, "missing coordinates or fna file"))
            continue

        df = pd.read_csv(coord_file, sep="\t", header=None, names=COORD_COLS)
        if df.empty:
            skipped.append((acc, "0 prophage regions (empty file)"))
            continue

        genome_records = {r.id: r for r in SeqIO.parse(fna_file, "fasta")}

        for i, row in df.iterrows():
            s, e = int(row["start"]), int(row["stop"])
            if s > e:
                s, e = e, s
            contig_id = row["contig"]
            if contig_id not in genome_records:
                matches = [k for k in genome_records if str(contig_id) in k or k in str(contig_id)]
                contig_id = matches[0] if matches else None
            if contig_id is None:
                skipped.append((acc, f"row {i}: could not resolve contig '{row['contig']}'"))
                continue

            seq = genome_records[contig_id].seq[s-1:e]
            header = f"{acc}__pp{i+1}__{s}-{e}"
            out_fh.write(f">{header}\n{str(seq)}\n")
            prophage_records.append({"accession": acc, "prophage_id": f"pp{i+1}",
                                      "contig": contig_id, "start": s, "end": e,
                                      "length": e - s + 1, "header": header})

meta_df = pd.DataFrame(prophage_records)
meta_df.to_csv(f"{OUT}/prophage_region_metadata.csv", index=False)
print(f"Extracted {len(meta_df)} prophage regions from {meta_df['accession'].nunique()} genomes")
if skipped:
    print(f"\n{len(skipped)} genomes/rows skipped:")
    for s in skipped:
        print(" ", s)

# Sanity check against the manuscript: should be 65 regions across 20 genomes (Figure 1 / Table 3)
print(f"\nExpected from manuscript: 65 regions / 20 genomes. Got: {len(meta_df)} regions / {meta_df['accession'].nunique()} genomes.")


## 6. Run geNomad (deep-learning viral classification)

geNomad scores each sequence for viral/plasmid/chromosomal signal using a neural network trained on a large marker-gene database — an independent check on PhiSpy's HMM/scoring-based calls. Runs `end-to-end` mode (annotation → marker classification → NN classification → score calibration).


In [ ]:
genomad_out = f"{WORK}/genomad_out"
os.makedirs(genomad_out, exist_ok=True)

!genomad end-to-end --cleanup --splits 4 {WORK}/prophage_regions.fna {genomad_out} {genomad_db}


In [ ]:
import glob, shutil, re
# geNomad produces separate viral and plasmid summary tables. Normalize the sequence
# identifier because some geNomad outputs append a |provirus_<id> suffix.
def normalize_header(x):
    if pd.isna(x):
        return x
    return str(x).strip().split("|provirus_", 1)[0]

summary_dirs = glob.glob(f"{genomad_out}/*_summary")
assert summary_dirs, "geNomad summary folder not found -- check the run log above for errors"
genomad_summary_dir = summary_dirs[0]
virus_files = glob.glob(f"{genomad_summary_dir}/*_virus_summary.tsv")
plasmid_files = glob.glob(f"{genomad_summary_dir}/*_plasmid_summary.tsv")
virus_raw = pd.read_csv(virus_files[0], sep="\t") if virus_files else pd.DataFrame()
plasmid_raw = pd.read_csv(plasmid_files[0], sep="\t") if plasmid_files else pd.DataFrame()

# Build a classification table covering all PhiSpy regions.
virus_raw["_merge_header"] = virus_raw["seq_name"].map(normalize_header) if not virus_raw.empty else pd.Series(dtype=str)
plasmid_raw["_merge_header"] = plasmid_raw["seq_name"].map(normalize_header) if not plasmid_raw.empty else pd.Series(dtype=str)
virus_headers = set(virus_raw["_merge_header"]) if not virus_raw.empty else set()
plasmid_headers = set(plasmid_raw["_merge_header"]) if not plasmid_raw.empty else set()

genomad_df = virus_raw.copy()
print(f"geNomad viral calls: {len(virus_headers)}")
print(f"geNomad plasmid calls: {len(plasmid_headers)}")

# Keep the raw outputs for reproducibility.
shutil.copytree(genomad_summary_dir, f"{OUT}/genomad_summary", dirs_exist_ok=True)


## 7. Run CheckV (completeness / contamination / quality tier)

This directly answers the manuscript's flagged gap: whether each PhiSpy region looks like a **complete**, **high-quality**, or **fragmented** prophage, versus a short/degenerate remnant.


In [ ]:
checkv_out = f"{WORK}/checkv_out"
!checkv end_to_end {WORK}/prophage_regions.fna {checkv_out} -d {checkv_db} -t 4

checkv_df = pd.read_csv(f"{checkv_out}/quality_summary.tsv", sep="\t")
print(checkv_df["checkv_quality"].value_counts())
display(checkv_df.head())

import shutil
shutil.copy(f"{checkv_out}/quality_summary.tsv", f"{OUT}/checkv_quality_summary.tsv")


## 8. Extract prophage-encoded proteins from the NCBI annotation

Pulls every CDS translation from `genomic.gbff` whose coordinates fall (fully or mostly) inside a called prophage region — no need to re-run gene prediction, since the NCBI RefSeq/GenBank annotation already has curated CDS + translations.


In [ ]:
from Bio import SeqIO as SeqIO2

OVERLAP_FRACTION = 0.5  # a CDS counts as "in" the prophage if >=50% of it overlaps the region

protein_records = []
protein_fasta_path = f"{WORK}/prophage_proteins.faa"

with open(protein_fasta_path, "w") as out_fh:
    for acc, sub in meta_df.groupby("accession"):
        gbff_file = f"{GENOMES_DIR}/{acc}/genomic.gbff"
        if not os.path.exists(gbff_file):
            continue
        gb_records = {r.id: r for r in SeqIO2.parse(gbff_file, "genbank")}

        for _, region in sub.iterrows():
            contig, rs, re_ = region["contig"], region["start"], region["end"]
            if contig not in gb_records:
                continue
            gb_rec = gb_records[contig]
            for feat in gb_rec.features:
                if feat.type != "CDS":
                    continue
                fs, fe = int(feat.location.start) + 1, int(feat.location.end)
                overlap = max(0, min(fe, re_) - max(fs, rs))
                if overlap / (fe - fs + 1) < OVERLAP_FRACTION:
                    continue
                prot_seq = feat.qualifiers.get("translation", [None])[0]
                if not prot_seq:
                    continue
                locus_tag = feat.qualifiers.get("locus_tag", ["NA"])[0]
                product = feat.qualifiers.get("product", ["NA"])[0]
                pid = f"{region['header']}__{locus_tag}"
                out_fh.write(f">{pid}\n{prot_seq}\n")
                protein_records.append({"protein_id": pid, "accession": acc,
                                         "prophage_id": region["prophage_id"],
                                         "locus_tag": locus_tag, "product": product,
                                         "length_aa": len(prot_seq)})

prot_df = pd.DataFrame(protein_records)
prot_df.to_csv(f"{OUT}/prophage_protein_metadata.csv", index=False)
print(f"Extracted {len(prot_df)} candidate prophage-encoded proteins")


## 9. ESM2 protein-language-model embeddings (GPU)

Every prophage-encoded protein is embedded with ESM2 and mean-pooled across residues. The resulting vectors are used for two reproducible downstream tasks: (i) putative functional annotation of proteins originally labelled as hypothetical, by nearest-neighbour similarity to informative proteins in the same dataset; and (ii) protein-content relatedness among independently validated prophage regions.

**Important:** these are sequence/embedding-based candidate annotations. They are not experimental functional assignments, and the embeddings are mean-centered before cosine-similarity comparisons to reduce anisotropy.

In [ ]:
import esm
import torch
import numpy as np

MODEL_NAME = "esm2_t12_35M_UR50D"
model, alphabet = esm.pretrained.load_model_and_alphabet(MODEL_NAME)
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
batch_converter = alphabet.get_batch_converter()
NUM_LAYERS = model.num_layers


In [ ]:
# (Re-load sequences cleanly from the FASTA we just wrote, sorted by length for efficient batching)
from Bio import SeqIO as SeqIO3

records = list(SeqIO3.parse(protein_fasta_path, "fasta"))
records.sort(key=lambda r: len(r.seq))

BATCH_SIZE = 16
CHECKPOINT_EVERY = 20  # batches

embeddings = {}
ckpt_path = f"{OUT}/esm2_embeddings.npz"

# resume support: load any existing checkpoint
if os.path.exists(ckpt_path):
    prev = np.load(ckpt_path, allow_pickle=True)
    embeddings = {k: prev[k] for k in prev.files}
    print(f"Resuming -- {len(embeddings)} embeddings already computed")

todo = [r for r in records if r.id not in embeddings]
print(f"{len(todo)} proteins left to embed")

for b_start in range(0, len(todo), BATCH_SIZE):
    batch = todo[b_start:b_start + BATCH_SIZE]
    data = [(r.id, str(r.seq)[:1022]) for r in batch]  # ESM2 practical length cap
    labels, strs, tokens = batch_converter(data)
    tokens = tokens.to(device)
    with torch.no_grad():
        out = model(tokens, repr_layers=[NUM_LAYERS], return_contacts=False)
    reps = out["representations"][NUM_LAYERS]
    for i, (pid, seq) in enumerate(data):
        rep = reps[i, 1:len(seq) + 1].mean(0).cpu().numpy()
        embeddings[pid] = rep

    if (b_start // BATCH_SIZE) % CHECKPOINT_EVERY == 0:
        np.savez(ckpt_path, **embeddings)
        print(f"  checkpoint: {len(embeddings)} / {len(records)} embedded")

np.savez(ckpt_path, **embeddings)
print(f"Done. {len(embeddings)} embeddings saved to {ckpt_path}")


## 10. Anisotropy-corrected ESM2 annotation

Raw ESM2 embeddings can exhibit anisotropy, which compresses cosine similarities into a narrow high-similarity range. We therefore subtract the global mean vector across all embedded proteins before nearest-neighbour comparison. Each protein originally annotated as **hypothetical protein** is assigned the product annotation of its nearest informative neighbour in the same dataset. The predefined high-confidence threshold is cosine similarity ≥0.80.

In [ ]:
from numpy.linalg import norm

protein_ids = list(embeddings.keys())
X = np.vstack([embeddings[p] for p in protein_ids])
global_mean = X.mean(axis=0)
centered = X - global_mean
centered_map = dict(zip(protein_ids, centered))

prot_meta = pd.DataFrame(protein_records)
prot_meta["is_hypothetical"] = prot_meta["product"].fillna("").str.lower().eq("hypothetical protein")
prot_meta["protein_id"] = prot_meta["protein_id"].astype(str)

informative = prot_meta[~prot_meta["is_hypothetical"]].copy()
informative = informative[informative["product"].fillna("").str.lower().ne("na")]
informative_ids = [p for p in informative["protein_id"] if p in centered_map]
inf_vecs = np.vstack([centered_map[p] for p in informative_ids])
inf_vecs = inf_vecs / (norm(inf_vecs, axis=1, keepdims=True) + 1e-9)
inf_products = dict(zip(informative["protein_id"], informative["product"]))
inf_prophages = dict(zip(informative["protein_id"], informative["prophage_id"]))

calls = []
for _, row in prot_meta[prot_meta["is_hypothetical"]].iterrows():
    pid = row["protein_id"]
    if pid not in centered_map or len(informative_ids) == 0:
        continue
    v = centered_map[pid]
    v = v / (norm(v) + 1e-9)
    sims = inf_vecs @ v
    idx = int(np.argmax(sims))
    score = float(sims[idx])
    nn_pid = informative_ids[idx]
    confidence = "high" if score >= 0.80 else ("medium" if score >= 0.70 else "low")
    calls.append({
        "protein_id": pid,
        "accession": row["accession"],
        "prophage_id": row["prophage_id"],
        "original_product": row["product"],
        "length_aa": row["length_aa"],
        "putative_function": inf_products[nn_pid],
        "nearest_neighbor_protein_id": nn_pid,
        "nearest_neighbor_prophage_id": inf_prophages.get(nn_pid, "NA"),
        "cosine_similarity": round(score, 4),
        "confidence": confidence,
    })

func_df = pd.DataFrame(calls)
func_df.to_csv(f"{OUT}/esm2_putative_function_calls.csv", index=False)
print(f"Hypothetical proteins: {prot_meta['is_hypothetical'].sum()}")
print(f"Resolved by nearest informative neighbour: {len(func_df)}")
print(func_df["confidence"].value_counts(dropna=False) if not func_df.empty else "No calls generated")


In [ ]:
# Summarize putative phage-associated functional categories from the transferred annotations.
PHAGE_TERMS = (
    "terminase", "capsid", "portal", "tail", "head", "holin", "lysin", "lysis",
    "integrase", "recombinase", "phage", "prophage", "hypothetical", "virion"
)
if not func_df.empty:
    func_df["phage_mobility_associated"] = func_df["putative_function"].fillna("").str.lower().apply(
        lambda x: any(term in x for term in PHAGE_TERMS)
    )
    informative_count = int(func_df["phage_mobility_associated"].sum())
    print(f"Putative phage/structural/mobility-associated transferred annotations: {informative_count}")
print("Interpret ESM2 transfers as candidate annotations, not experimentally validated functions.")


## 11. Merge everything into one master table
Joins PhiSpy coordinates + geNomad classification + CheckV quality + ESM2 cluster/function calls, keyed on the shared `header`/`protein_id` scheme from Cell 5/8.

In [ ]:
# 11. Merge everything into one master table

def normalize_header(x):
    if pd.isna(x):
        return x
    return str(x).strip().split("|provirus_", 1)[0]

master = meta_df.copy()
master["_merge_header"] = master["header"].map(normalize_header)

# geNomad: combine viral and plasmid summaries; remaining PhiSpy calls are unresolved.
virus_raw = virus_raw.copy() if "virus_raw" in globals() else pd.DataFrame()
plasmid_raw = plasmid_raw.copy() if "plasmid_raw" in globals() else pd.DataFrame()

def prep_genomad(df, call):
    if df.empty:
        return pd.DataFrame(columns=["_merge_header", "genomad_call"])
    x = df.copy()
    x["_merge_header"] = x["seq_name"].map(normalize_header)
    x["genomad_call"] = call
    return x

gv = prep_genomad(virus_raw, "virus")
gp = prep_genomad(plasmid_raw, "plasmid")

if not gv.empty:
    gv = gv.add_prefix("genomad_").rename(columns={"genomad__merge_header": "_merge_header"})
    master = master.merge(gv, on="_merge_header", how="left")
if not gp.empty:
    gp = gp.add_prefix("genomad_plasmid_").rename(columns={"genomad_plasmid__merge_header": "_merge_header"})
    master = master.merge(gp, on="_merge_header", how="left")

master["genomad_call"] = master.get("genomad_genomad_call", pd.Series(index=master.index, dtype=object))
# If both tables are present, the plasmid classification fills only rows not already viral.
if "genomad_plasmid_genomad_call" in master:
    master["genomad_call"] = master["genomad_call"].fillna(master["genomad_plasmid_genomad_call"])
master["genomad_call"] = master["genomad_call"].fillna("unclassified")
master["genomad_confirmed"] = master["genomad_call"].eq("virus")

# CheckV uses the same canonical sequence identifier.
checkv = checkv_df.rename(columns={checkv_df.columns[0]: "header"}).copy()
checkv["_merge_header"] = checkv["header"].map(normalize_header)
checkv = checkv.add_prefix("checkv_").rename(columns={"checkv__merge_header": "_merge_header"})
master = master.merge(checkv, on="_merge_header", how="left")

# Protein-level summaries.
protein_summary = prot_df.groupby(["accession", "prophage_id"]).agg(
    n_proteins=("protein_id", "count"),
    n_hypothetical=("product", lambda x: x.fillna("").str.lower().eq("hypothetical protein").sum()),
).reset_index()
protein_summary["pct_hypothetical"] = (protein_summary["n_hypothetical"] / protein_summary["n_proteins"] * 100).round(1)
master = master.merge(protein_summary, on=["accession", "prophage_id"], how="left")

if not func_df.empty:
    esm2_summary = func_df.groupby(["accession", "prophage_id"]).agg(
        n_resolved_high=("confidence", lambda x: (x == "high").sum()),
        n_resolved_medium=("confidence", lambda x: (x == "medium").sum()),
        n_resolved_low=("confidence", lambda x: (x == "low").sum()),
    ).reset_index()
    master = master.merge(esm2_summary, on=["accession", "prophage_id"], how="left")

for col in ["n_resolved_high", "n_resolved_medium", "n_resolved_low"]:
    if col not in master:
        master[col] = 0
    master[col] = master[col].fillna(0).astype(int)

master = master.drop(columns=["_merge_header", "genomad_genomad_call", "genomad_plasmid_genomad_call"], errors="ignore")
master.to_csv(f"{OUT}/MASTER_prophage_validation_table.csv", index=False)
print(f"Master table: {master.shape[0]} rows x {master.shape[1]} columns")
print(master["genomad_call"].value_counts(dropna=False).to_dict())
assert len(master) == 65
assert master["genomad_call"].eq("virus").sum() == 44
assert master["genomad_call"].eq("plasmid").sum() == 14
assert master["genomad_call"].eq("unclassified").sum() == 7


## 12. Summary figures for the manuscript

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

checkv_df["checkv_quality"].value_counts().plot(kind="bar", ax=axes[0], color="#2b6cb0")
axes[0].set_title("CheckV quality tier across predicted prophage regions")
axes[0].set_ylabel("Number of regions")
axes[0].set_xlabel("CheckV quality tier")

scatter = axes[1].scatter(cluster_df["umap_x"], cluster_df["umap_y"],
                           c=cluster_df["cluster"], cmap="tab20", s=12)
axes[1].set_title("ESM2 embedding space of prophage-encoded proteins")
axes[1].set_xlabel("UMAP-1"); axes[1].set_ylabel("UMAP-2")

plt.tight_layout()
plt.savefig(f"{OUT}/Figure_GPU_prophage_validation.png", dpi=300)
plt.show()


## 13. Download your results

All outputs are in `/content/outputs/` on the Colab machine (this disk is wiped when the session ends, so download now):

| File | Manuscript use |
|---|---|
| `MASTER_prophage_validation_table.csv` | New supplementary table: PhiSpy call cross-validated by geNomad + CheckV quality tier per region |
| `esm2_putative_function_calls.csv` | Cargo-gene functional annotation (integrase/lysin/holin/RBP calls) — feeds the mechanistic Discussion paragraphs on DF, DSM 17938 lysogeny modules |
| `esm2_clusters.csv` | Protein family clusters — useful for spotting cargo genes shared across strains (host-phage ecology angle) |
| `Figure_GPU_prophage_validation.png` | Candidate new figure (CheckV quality distribution + ESM2 embedding map) |
| `checkv_quality_summary.tsv`, `genomad_summary/` | Raw tool outputs, for methods reporting / reproducibility |

Run the next cell to zip everything and trigger a browser download.


In [ ]:
import shutil
from google.colab import files as colab_files

zip_path = "/content/GPU_Prophage_Analysis_results"
shutil.make_archive(zip_path, "zip", OUT)
print(f"Zipped -> {zip_path}.zip")
colab_files.download(f"{zip_path}.zip")


**Next step once this has run:** bring `MASTER_prophage_validation_table.csv` and `esm2_putative_function_calls.csv` back to me (just upload the zip, or those two files) and I'll fold the results into a new Methods subsection (2.9), a new Results subsection, the new figure, and the Discussion/Conclusion rewrites we scoped — including finally reconciling the spacer–protospacer (Table 7) framing with this new completeness/functional evidence.
